# Interactive Chat while tracking Persona Drift

This notebook provides an interactive chat interface that tracks the projection
of model activations onto the assistant axis.

In [ ]:
import os
import sys
from pathlib import Path

# Make imports and data paths work whether the notebook is run from the repo
# root (nbconvert) or from the notebooks/ directory (Jupyter UI).
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn.functional as F
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from huggingface_hub import hf_hub_download

from assistant_axis import (
    load_axis,
    get_config,
)


## Load Model and Axis

In [ ]:
# Configuration
# Set ASSISTANT_AXIS_SMOKE_TEST=1 to run the notebook end-to-end without
# downloading the gated 70B model. Smoke mode exercises notebook plumbing only.
SMOKE_TEST = os.environ.get("ASSISTANT_AXIS_SMOKE_TEST", "0") == "1"

if SMOKE_TEST:
    MODEL_NAME = "hf-internal-testing/tiny-random-LlamaForCausalLM"
    MODEL_SHORT = "smoke-test"
    TARGET_LAYER = 0
    REPO_ID = None
else:
    MODEL_NAME = "meta-llama/Llama-3.3-70B-Instruct"
    MODEL_SHORT = "llama-3.3-70b"
    REPO_ID = "lu-christina/assistant-axis-vectors"

    # Get model config
    config = get_config(MODEL_NAME)
    TARGET_LAYER = config["target_layer"]

TRANSCRIPT_PATH = PROJECT_ROOT / "transcripts" / "case_studies" / MODEL_SHORT / "selfharm_unsteered.json"
if SMOKE_TEST:
    TRANSCRIPT_PATH = PROJECT_ROOT / "transcripts" / "case_studies" / "llama-3.3-70b" / "selfharm_unsteered.json"

print(f"Model: {MODEL_NAME}")
print(f"Target layer: {TARGET_LAYER}")
print(f"Transcript: {TRANSCRIPT_PATH}")


In [ ]:
# Load model using ProbingModel wrapper
from types import SimpleNamespace
from assistant_axis.internals import ProbingModel

if SMOKE_TEST:
    print("Smoke-test mode: using a lightweight placeholder model.")
    pm = SimpleNamespace(hidden_size=16, model=None, tokenizer=None)
else:
    print("Loading model...")
    try:
        pm = ProbingModel(MODEL_NAME)
    except OSError as exc:
        raise RuntimeError(
            "Could not load the configured Hugging Face model. "
            "For the default Llama 3.3 70B notebook run, make sure your Hugging Face "
            "account has access to meta-llama/Llama-3.3-70B-Instruct and that you are "
            "authenticated in this environment. For a quick local plumbing check, run "
            "with ASSISTANT_AXIS_SMOKE_TEST=1."
        ) from exc

model = pm.model
tokenizer = pm.tokenizer
print("Model loaded!")


In [ ]:
# Load axis from HuggingFace, or build a random axis in smoke-test mode.
if SMOKE_TEST:
    axis_vec = F.normalize(torch.randn(pm.hidden_size), dim=0)
    print(f"Smoke-test axis vector shape: {axis_vec.shape}")
else:
    axis_path = hf_hub_download(repo_id=REPO_ID, filename=f"{MODEL_SHORT}/assistant_axis.pt", repo_type="dataset")
    axis = load_axis(axis_path)
    print(f"Axis shape: {axis.shape}")

    # Normalize
    axis_vec = F.normalize(axis[TARGET_LAYER].float(), dim=0)
    print(f"Axis vector shape: {axis_vec.shape}")


## Load and Project Transcript

Load a conversation transcript and compute projections for each assistant turn.

In [ ]:
import json
import matplotlib.pyplot as plt

from assistant_axis.internals import ConversationEncoder, SpanMapper


def load_transcript(filepath):
    """Load a conversation transcript from JSON."""
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"Transcript not found: {filepath}")

    with filepath.open() as f:
        data = json.load(f)
    return data['conversation']


def compute_trajectory(pm, conversation, axis_vec, layer):
    """
    Compute projection for each assistant turn in a conversation.
    Uses single forward pass and span mapping (matches fig1_trajectory.ipynb approach).
    """
    if SMOKE_TEST:
        # Keep smoke-test execution local and fast; these values are not meaningful.
        num_assistant_turns = sum(turn.get("role") == "assistant" for turn in conversation)
        torch.manual_seed(0)
        return torch.randn(num_assistant_turns).numpy()

    encoder = ConversationEncoder(pm.tokenizer)
    span_mapper = SpanMapper(pm.tokenizer)

    # Get mean activations for all turns (single forward pass)
    mean_acts = span_mapper.mean_all_turn_activations(pm, encoder, conversation, layer=layer)

    # mean_acts shape: (num_turns, hidden_size)
    # Filter to assistant turns only (odd indices: 1, 3, 5, ...)
    assistant_acts = mean_acts[1::2].float()

    # Project onto axis (dot product)
    projections = (assistant_acts @ axis_vec.cpu()).numpy()

    return projections


def plot_trajectory(projections, proj_capped=None, title="Projection Trajectory"):
    """Plot projection trajectory, optionally with activation capped line."""
    plt.figure(figsize=(8, 5))
    turns = range(len(projections))

    plt.plot(turns, projections, marker='o', label='Unsteered')

    if proj_capped is not None:
        plt.plot(range(len(proj_capped)), proj_capped, marker='o', alpha=0.6, label='Activation Capped')
        plt.legend()

    plt.xlabel('Conversation Turn (assistant)')
    plt.ylabel('Projection on Assistant Axis')
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Load transcript and compute trajectory
conversation = load_transcript(TRANSCRIPT_PATH)
print(f"Loaded {len(conversation)} turns ({len(conversation)//2} assistant responses)")

projections = compute_trajectory(pm, conversation, axis_vec, TARGET_LAYER)
print(f"Computed {len(projections)} projections")
print(f"Range: [{projections.min():.2f}, {projections.max():.2f}]")

# Plot trajectory
plot_trajectory(projections, title="Projection Trajectory")
